# Train Chatbot NLP - Quản lý công việc (LSTM + PhoBERT)

Notebook này train 2 model phân loại intent cho chatbot tiếng Việt, sau đó kết hợp thành hệ thống 3 tầng.

**Dataset**: `task_management_chatbot_dataset.json`

**Kiến trúc 3 tầng:**
```
User input
  ├─→ Tầng 1: LSTM (model tự train) → nếu confidence > 85% → dùng
  ├─→ Tầng 2: PhoBERT (fine-tune)   → nếu confidence > 60% → dùng
  └─→ Tầng 3: Hỏi lại user
```

**Các bước:**
1. Cài đặt thư viện & Load dataset
2. Train Model LSTM
3. Train Model PhoBERT
4. So sánh LSTM vs PhoBERT
5. Hệ thống 3 tầng & Test
6. Lưu model & tải về

## 1. Cài đặt thư viện

In [ ]:
!pip install underthesea transformers -q

In [ ]:
import json
import re
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from underthesea import word_tokenize
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Kiểm tra GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Load Dataset

In [ ]:
# Upload file dataset lên Colab
from google.colab import files
print("Hãy upload file 'task_management_chatbot_dataset.json':")
uploaded = files.upload()

In [ ]:
# Load dataset
with open('task_management_chatbot_dataset.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)

train_data = dataset['training_data']
test_data = dataset['test_data']

# Lấy danh sách tags (categories)
tags = sorted(list(set(d['category'] for d in train_data)))
tag2idx = {tag: idx for idx, tag in enumerate(tags)}

# Tạo responses dict từ dataset
responses_dict = {}
for d in train_data:
    tag = d['category']
    if tag not in responses_dict:
        responses_dict[tag] = []
    if d['expected_output'] not in responses_dict[tag]:
        responses_dict[tag].append(d['expected_output'])

print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")
print(f"Số categories (intents): {len(tags)}")
print(f"\nDanh sách intents:")
for i, tag in enumerate(tags):
    count_train = sum(1 for d in train_data if d['category'] == tag)
    count_test = sum(1 for d in test_data if d['category'] == tag)
    print(f'  {i:2d}. {tag:30s} {count_train:3d} train / {count_test:2d} test')

## 3. Tiền xử lý dữ liệu

In [ ]:
def tokenize_vi(text):
    """Tách từ tiếng Việt sử dụng underthesea."""
    text = text.lower().strip()
    text = re.sub(r"[^\w\sàáảãạăắằẳẵặâấầẩẫậèéẻẽẹêếềểễệìíỉĩịòóỏõọôốồổỗộơớờởỡợùúủũụưứừửữựỳýỷỹỵđ]", "", text)
    tokens = word_tokenize(text, format='text').split()
    return tokens

# Test tách từ
test_sentences = [
    "thêm công việc mới cho tôi",
    "nhắc tôi lúc 3 giờ chiều",
    "hôm nay tôi có gì cần làm",
]
for s in test_sentences:
    print(f'{s} -> {tokenize_vi(s)}')

In [ ]:
# Xây dựng vocabulary từ cả training + test data
all_words = set()
for d in train_data + test_data:
    tokens = tokenize_vi(d['user_input'])
    all_words.update(tokens)

all_words = sorted(all_words)
word2idx = {word: idx + 1 for idx, word in enumerate(all_words)}  # 0 = padding

print(f'Vocabulary size: {len(word2idx)}')
print(f'Tags ({len(tags)}): {tags}')
print(f'\nMột số từ trong vocab: {all_words[:20]}')

In [ ]:
MAX_LEN = 30

def text_to_sequence(text, word2idx, max_len=MAX_LEN):
    """Chuyển text thành sequence số."""
    tokens = tokenize_vi(text)
    sequence = [word2idx.get(token, 0) for token in tokens]
    if len(sequence) < max_len:
        sequence = sequence + [0] * (max_len - len(sequence))
    else:
        sequence = sequence[:max_len]
    return sequence

# Chuẩn bị dữ liệu TRAINING
X_train, y_train = [], []
for d in train_data:
    X_train.append(text_to_sequence(d['user_input'], word2idx))
    y_train.append(tag2idx[d['category']])
X_train = np.array(X_train)
y_train = np.array(y_train)

# Chuẩn bị dữ liệu TEST
X_test, y_test = [], []
for d in test_data:
    X_test.append(text_to_sequence(d['user_input'], word2idx))
    y_test.append(tag2idx[d['category']])
X_test = np.array(X_test)
y_test = np.array(y_test)

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}, y_test:  {y_test.shape}')

lengths = [len(tokenize_vi(d['user_input'])) for d in train_data + test_data]
print(f'\nĐộ dài câu: min={min(lengths)}, max={max(lengths)}, avg={np.mean(lengths):.1f}')

## 4. Phân bố dữ liệu

In [ ]:
intent_counts = [sum(1 for d in train_data if d['category'] == tag) for tag in tags]

plt.figure(figsize=(16, 6))
bars = plt.bar(range(len(tags)), intent_counts, color='steelblue')
plt.xlabel('Intent')
plt.ylabel('Số lượng training samples')
plt.title(f'Phân bố training samples theo Intent ({len(tags)} categories)')
plt.xticks(range(len(tags)), tags, rotation=70, ha='right', fontsize=8)
for bar, count in zip(bars, intent_counts):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
             str(count), ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.show()

## 5. Xây dựng Model LSTM

In [ ]:
class ChatbotLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim,
                 num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0,
                            bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hidden, _) = self.lstm(embedded)
        hidden_fwd = hidden[-2]
        hidden_bwd = hidden[-1]
        hidden_cat = torch.cat((hidden_fwd, hidden_bwd), dim=1)
        output = self.dropout(hidden_cat)
        output = self.fc(output)
        return output

# Hyperparameters
VOCAB_SIZE = len(word2idx) + 1
EMBEDDING_DIM = 256
HIDDEN_DIM = 256
OUTPUT_DIM = len(tags)
NUM_LAYERS = 2
DROPOUT = 0.3

model = ChatbotLSTM(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM,
                    NUM_LAYERS, DROPOUT).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nTổng parameters: {total_params:,}')

## 6. Training

In [ ]:
class IntentDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.LongTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

BATCH_SIZE = 32
EPOCHS = 300
LEARNING_RATE = 0.001

train_dataset = IntentDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_dataset = IntentDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

print(f'Training: {len(train_dataset)}, Test: {len(test_dataset)}, Epochs: {EPOCHS}\n')

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        output = model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(output, 1)
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)

    train_loss = total_loss / len(train_loader)
    train_acc = correct / total * 100
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)

    model.eval()
    test_loss_total, test_correct, test_total = 0, 0, 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            output = model(batch_X)
            loss = criterion(output, batch_y)
            test_loss_total += loss.item()
            _, predicted = torch.max(output, 1)
            test_correct += (predicted == batch_y).sum().item()
            test_total += batch_y.size(0)

    test_loss = test_loss_total / len(test_loader)
    test_acc = test_correct / test_total * 100
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)

    if (epoch + 1) % 30 == 0:
        print(f'Epoch [{epoch+1}/{EPOCHS}]  Train Loss: {train_loss:.4f} Acc: {train_acc:.1f}%  |  Test Loss: {test_loss:.4f} Acc: {test_acc:.1f}%')

print(f'\nFinal Train — Loss: {history["train_loss"][-1]:.4f}, Acc: {history["train_acc"][-1]:.1f}%')
print(f'Final Test  — Loss: {history["test_loss"][-1]:.4f}, Acc: {history["test_acc"][-1]:.1f}%')

## 7. Visualize kết quả Training

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history['train_loss'], color='red', linewidth=1.5, label='Train Loss')
ax1.plot(history['test_loss'], color='orange', linewidth=1.5, label='Test Loss', linestyle='--')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Training & Test Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history['train_acc'], color='blue', linewidth=1.5, label='Train Accuracy')
ax2.plot(history['test_acc'], color='green', linewidth=1.5, label='Test Accuracy', linestyle='--')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.set_title('Training & Test Accuracy')
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 8. Đánh giá Model trên tập Test

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        output = model(batch_X)
        _, predicted = torch.max(output, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(batch_y.numpy())

test_tags_idx = sorted(list(set(all_labels)))
test_tag_names = [tags[i] for i in test_tags_idx]

print('=== Classification Report (Test Set) ===\n')
print(classification_report(all_labels, all_preds, labels=test_tags_idx,
                            target_names=test_tag_names, zero_division=0))

cm = confusion_matrix(all_labels, all_preds, labels=test_tags_idx)
plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=test_tag_names, yticklabels=test_tag_names)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix (Test Set)')
plt.xticks(rotation=60, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout(); plt.show()

print('\n=== Các trường hợp dự đoán SAI ===\n')
wrong = 0
for i, (true_l, pred_l) in enumerate(zip(all_labels, all_preds)):
    if true_l != pred_l:
        wrong += 1
        print(f'  Input: "{test_data[i]["user_input"]}"')
        print(f'  True: {tags[true_l]}  |  Pred: {tags[pred_l]}\n')
if wrong == 0:
    print('  Không có trường hợp nào dự đoán sai!')
else:
    print(f'Tổng sai: {wrong}/{len(all_labels)} ({wrong/len(all_labels)*100:.1f}%)')

## PHẦN 2: Fine-tune PhoBERT

PhoBERT là pre-trained model cho tiếng Việt, hiểu ngữ cảnh tốt hơn LSTM.
Ta sẽ fine-tune PhoBERT trên cùng dataset rồi so sánh.

In [ ]:
from transformers import AutoTokenizer, AutoModel

# Load PhoBERT tokenizer và model
phobert_name = "vinai/phobert-base"
phobert_tokenizer = AutoTokenizer.from_pretrained(phobert_name)
phobert_base = AutoModel.from_pretrained(phobert_name)

print(f"PhoBERT loaded: {phobert_name}")
print(f"Hidden size: {phobert_base.config.hidden_size}")

# Test tokenize
test = "giúp tôi thêm công việc hoàn thành bài tập"
tokens = phobert_tokenizer(test, return_tensors="pt")
print(f"\nTest: '{test}'")
print(f"Tokens: {phobert_tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])}")

In [ ]:
# PhoBERT Classifier
class PhoBERTClassifier(nn.Module):
    def __init__(self, phobert_base, num_classes, dropout=0.3):
        super().__init__()
        self.phobert = phobert_base
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(phobert_base.config.hidden_size, num_classes)

        # Freeze PhoBERT layers (chỉ train FC layer đầu tiên, sau đó unfreeze dần)
        for param in self.phobert.parameters():
            param.requires_grad = False

    def unfreeze_last_layers(self, n=2):
        """Unfreeze n layers cuối của PhoBERT để fine-tune."""
        for param in self.phobert.encoder.layer[-n:].parameters():
            param.requires_grad = True
        print(f"Unfreezed last {n} layers of PhoBERT")

    def forward(self, input_ids, attention_mask):
        outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        # Lấy [CLS] token
        cls_output = outputs.last_hidden_state[:, 0, :]
        output = self.dropout(cls_output)
        output = self.fc(output)
        return output

phobert_model = PhoBERTClassifier(phobert_base, len(tags)).to(device)

total_params = sum(p.numel() for p in phobert_model.parameters())
trainable_params = sum(p.numel() for p in phobert_model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")

In [ ]:
# Dataset cho PhoBERT
class PhoBERTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Chuẩn bị data
train_texts = [d['user_input'] for d in train_data]
train_labels = [tag2idx[d['category']] for d in train_data]
test_texts = [d['user_input'] for d in test_data]
test_labels = [tag2idx[d['category']] for d in test_data]

phobert_train_dataset = PhoBERTDataset(train_texts, train_labels, phobert_tokenizer)
phobert_test_dataset = PhoBERTDataset(test_texts, test_labels, phobert_tokenizer)
phobert_train_loader = DataLoader(phobert_train_dataset, batch_size=16, shuffle=True)
phobert_test_loader = DataLoader(phobert_test_dataset, batch_size=16, shuffle=False)

print(f"PhoBERT Train: {len(phobert_train_dataset)}, Test: {len(phobert_test_dataset)}")

In [ ]:
# === Training PhoBERT ===
# Phase 1: Train FC layer only (5 epochs)
# Phase 2: Unfreeze last 2 layers + fine-tune (10 epochs)

phobert_criterion = nn.CrossEntropyLoss()
phobert_optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, phobert_model.parameters()),
    lr=2e-4, weight_decay=0.01
)

phobert_history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

def train_phobert_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        output = model(input_ids, attention_mask)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(output, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total * 100

def eval_phobert(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            output = model(input_ids, attention_mask)
            loss = criterion(output, labels)
            total_loss += loss.item()
            _, predicted = torch.max(output, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total * 100

# Phase 1: Train FC only
print("=== Phase 1: Train FC layer (5 epochs) ===\n")
for epoch in range(5):
    train_loss, train_acc = train_phobert_epoch(phobert_model, phobert_train_loader, phobert_criterion, phobert_optimizer)
    test_loss, test_acc = eval_phobert(phobert_model, phobert_test_loader, phobert_criterion)
    phobert_history['train_loss'].append(train_loss)
    phobert_history['train_acc'].append(train_acc)
    phobert_history['test_loss'].append(test_loss)
    phobert_history['test_acc'].append(test_acc)
    print(f"Epoch {epoch+1}/5  Train: {train_loss:.4f} {train_acc:.1f}%  |  Test: {test_loss:.4f} {test_acc:.1f}%")

# Phase 2: Unfreeze last 2 layers
print("\n=== Phase 2: Fine-tune last 2 layers (10 epochs) ===\n")
phobert_model.unfreeze_last_layers(2)
phobert_optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, phobert_model.parameters()),
    lr=2e-5, weight_decay=0.01
)

for epoch in range(10):
    train_loss, train_acc = train_phobert_epoch(phobert_model, phobert_train_loader, phobert_criterion, phobert_optimizer)
    test_loss, test_acc = eval_phobert(phobert_model, phobert_test_loader, phobert_criterion)
    phobert_history['train_loss'].append(train_loss)
    phobert_history['train_acc'].append(train_acc)
    phobert_history['test_loss'].append(test_loss)
    phobert_history['test_acc'].append(test_acc)
    print(f"Epoch {epoch+1}/10  Train: {train_loss:.4f} {train_acc:.1f}%  |  Test: {test_loss:.4f} {test_acc:.1f}%")

print(f"\nPhoBERT Final — Train: {phobert_history['train_acc'][-1]:.1f}%, Test: {phobert_history['test_acc'][-1]:.1f}%")

## So sánh LSTM vs PhoBERT

In [ ]:
# Đánh giá PhoBERT trên test set
phobert_model.eval()
phobert_preds, phobert_labels = [], []
with torch.no_grad():
    for batch in phobert_test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        output = phobert_model(input_ids, attention_mask)
        _, predicted = torch.max(output, 1)
        phobert_preds.extend(predicted.cpu().numpy())
        phobert_labels.extend(batch['label'].numpy())

# So sánh
from sklearn.metrics import accuracy_score
lstm_acc = accuracy_score(all_labels, all_preds) * 100  # từ phần LSTM
phobert_acc = accuracy_score(phobert_labels, phobert_preds) * 100

print("=" * 50)
print(f"{'Model':<15} {'Test Accuracy':>15}")
print("=" * 50)
print(f"{'LSTM':<15} {lstm_acc:>14.1f}%")
print(f"{'PhoBERT':<15} {phobert_acc:>14.1f}%")
print("=" * 50)

# So sánh từng câu test
print("\n=== Các câu LSTM sai nhưng PhoBERT đúng ===\n")
for i in range(len(test_data)):
    lstm_pred = tags[all_preds[i]]
    phobert_pred = tags[phobert_preds[i]]
    true_label = tags[all_labels[i]]
    if lstm_pred != true_label and phobert_pred == true_label:
        print(f'  "{test_data[i]["user_input"]}"')
        print(f'  True: {true_label} | LSTM: {lstm_pred} | PhoBERT: {phobert_pred} ✅\n')

print("\n=== Các câu PhoBERT sai nhưng LSTM đúng ===\n")
for i in range(len(test_data)):
    lstm_pred = tags[all_preds[i]]
    phobert_pred = tags[phobert_preds[i]]
    true_label = tags[all_labels[i]]
    if phobert_pred != true_label and lstm_pred == true_label:
        print(f'  "{test_data[i]["user_input"]}"')
        print(f'  True: {true_label} | LSTM: {lstm_pred} ✅ | PhoBERT: {phobert_pred}\n')

# Bar chart so sánh
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['LSTM\n(tự train)', 'PhoBERT\n(fine-tune)'], [lstm_acc, phobert_acc],
              color=['steelblue', 'coral'])
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('So sánh LSTM vs PhoBERT')
ax.set_ylim(0, 105)
for bar, acc in zip(bars, [lstm_acc, phobert_acc]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{acc:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## Hệ thống 3 tầng: LSTM → PhoBERT → Hỏi lại

```
User input
  ├─→ Tầng 1: LSTM → nếu confidence > 85% → dùng (nhanh)
  ├─→ Tầng 2: PhoBERT → nếu confidence > 60% → dùng (chính xác)
  └─→ Tầng 3: Hỏi lại user
```

## 9. Test thử Chatbot

In [ ]:
def predict_lstm(text):
    """Tầng 1: LSTM model."""
    model.eval()
    sequence = text_to_sequence(text, word2idx)
    input_tensor = torch.LongTensor([sequence]).to(device)
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        confidence, predicted = torch.max(probs, 1)
    return tags[predicted.item()], confidence.item()

def predict_phobert(text):
    """Tầng 2: PhoBERT model."""
    phobert_model.eval()
    encoding = phobert_tokenizer(text, max_length=64, padding='max_length',
                                  truncation=True, return_tensors='pt')
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    with torch.no_grad():
        output = phobert_model(input_ids, attention_mask)
        probs = torch.softmax(output, dim=1)
        confidence, predicted = torch.max(probs, 1)
    return tags[predicted.item()], confidence.item()

def predict_3tier(text, lstm_threshold=0.85, phobert_threshold=0.60):
    """Hệ thống 3 tầng: LSTM → PhoBERT → Hỏi lại."""
    # Tầng 1: LSTM
    lstm_tag, lstm_conf = predict_lstm(text)
    if lstm_conf >= lstm_threshold:
        return lstm_tag, lstm_conf, 'LSTM'

    # Tầng 2: PhoBERT
    phobert_tag, phobert_conf = predict_phobert(text)
    if phobert_conf >= phobert_threshold:
        return phobert_tag, phobert_conf, 'PhoBERT'

    # Tầng 3: Chọn model có confidence cao hơn, hoặc hỏi lại
    if lstm_conf >= phobert_conf:
        return lstm_tag, lstm_conf, 'LSTM(low)'
    return phobert_tag, phobert_conf, 'PhoBERT(low)'

def extract_task_name(text, intent):
    """Trích xuất tên công việc từ câu nói."""
    text_lower = text.lower().strip()
    triggers_map = {
        'create_task': ['tạo công việc mới', 'tạo công việc:', 'tạo công việc', 'tạo task mới:', 'tạo task mới', 'tạo task:', 'tạo task',
                        'thêm công việc:', 'thêm công việc', 'thêm việc', 'thêm task', 'add task',
                        'giúp tôi thêm công việc', 'giúp tôi tạo'],
        'delete_task': ['xóa công việc', 'xóa việc', 'xóa task', 'hủy công việc', 'bỏ công việc',
                        'remove task', 'delete task'],
        'update_task': ['sửa công việc', 'cập nhật công việc', 'chỉnh sửa công việc', 'sửa task',
                        'edit task', 'update task', 'thay đổi công việc'],
        'status_update': ['hoàn thành công việc', 'hoàn thành task', 'hoàn thành việc', 'hoàn thành',
                          'đánh dấu hoàn thành', 'đánh dấu xong',
                          'done task', 'done', 'finish task', 'complete task', 'mark done',
                          'xong task', 'xong việc', 'xong',
                          'tôi đã hoàn thành', 'tôi đã xong', 'tôi đã làm xong',
                          'đã hoàn thành', 'đã làm xong', 'đã xong'],
        'reminder': ['nhắc tôi', 'nhắc nhở tôi', 'đặt nhắc nhở', 'nhắc giúp tôi'],
        'reschedule': ['dời công việc', 'dời task', 'lùi deadline', 'hoãn việc'],
        'add_note': ['thêm ghi chú cho', 'ghi chú cho', 'note cho'],
        'task_detail': ['chi tiết công việc', 'chi tiết task', 'xem chi tiết', 'thông tin task'],
        'assign_task': ['gán task', 'gán công việc', 'phân công', 'giao task'],
    }
    triggers = triggers_map.get(intent, [])
    triggers = sorted(triggers, key=len, reverse=True)
    for trigger in triggers:
        idx = text_lower.find(trigger)
        if idx != -1:
            after = text_lower[idx + len(trigger):].strip()
            for filler in ['là', ':', 'về', 'việc', 'cho tôi', 'giúp tôi', 'đi', 'task', 'công việc']:
                if after.startswith(filler + ' '):
                    after = after[len(filler):].strip()
                elif after == filler:
                    after = ''
            after = re.sub(r'\s*(lúc|vào)\s+\d{1,2}\s*(giờ|h).*$', '', after).strip()
            after = re.sub(r'\s*(ngày mai|hôm nay|chiều nay|sáng mai|tối nay|tuần sau).*$', '', after).strip()
            after = after.strip("'\"")
            if after:
                return after
    return None

RESPONSE_TEMPLATES = {
    'greeting': ["Xin chào! Tôi có thể giúp gì cho bạn?"],
    'goodbye': ["Tạm biệt! Chúc bạn làm việc hiệu quả!"],
    'thank_you': ["Không có gì! Tôi luôn sẵn sàng giúp bạn!"],
    'bot_identity': ["Tôi là chatbot quản lý công việc. Tôi giúp bạn thêm, sửa, xóa task, nhắc nhở và theo dõi tiến độ!"],
    'create_task': ["Đã tạo công việc '{task_name}'.", "Bạn muốn thêm công việc gì?"],
    'delete_task': ["Bạn có chắc muốn xóa '{task_name}' không?", "Bạn muốn xóa công việc nào?"],
    'update_task': ["Bạn muốn sửa gì cho '{task_name}'?", "Bạn muốn cập nhật công việc nào?"],
    'status_update': ["Đã đánh dấu '{task_name}' hoàn thành.", "Công việc nào đã hoàn thành?"],
    'list_tasks': ["Đây là danh sách công việc của bạn:"],
    'search_task': ["Để tôi tìm cho bạn."],
    'assign_task': ["Đã gán '{task_name}'.", "Bạn muốn gán task cho ai?"],
    'deadline_management': ["Bạn muốn đặt deadline cho task nào?"],
    'priority_management': ["Bạn muốn thay đổi ưu tiên task nào?"],
    'statistics': ["Đây là thống kê công việc:"],
    'reminder': ["Đã đặt nhắc nhở cho '{task_name}'.", "Bạn muốn nhắc nhở về việc gì?"],
    'collaboration': ["Bạn muốn chia sẻ với ai?"],
    'general_help': ["Tôi có thể giúp: thêm/sửa/xóa task, nhắc nhở, thống kê. Bạn cần gì?"],
    'complete_all_tasks': ["Bạn có chắc muốn đánh dấu hoàn thành TẤT CẢ? (Có/Không)"],
    'delete_all_tasks': ["⚠️ Bạn có chắc muốn XÓA TẤT CẢ? (Có/Không)"],
    'confirm_complete': ["Đã đánh dấu hoàn thành!"],
    'confirm_delete': ["Đã xóa!"],
    'task_today': ["Đây là công việc hôm nay:"],
    'task_overdue': ["Các công việc đã quá hạn:"],
    'task_upcoming': ["Công việc sắp tới:"],
    'task_by_category': ["Để tôi lọc cho bạn."],
    'sort_tasks': ["Để tôi sắp xếp lại."],
    'add_note': ["Đã thêm ghi chú cho '{task_name}'.", "Bạn muốn ghi chú gì?"],
    'task_detail': ["Chi tiết '{task_name}':", "Bạn muốn xem chi tiết task nào?"],
    'reschedule': ["Đã dời '{task_name}'.", "Bạn muốn dời task nào?"],
    'set_duration': ["Đã ghi nhận thời gian."],
    'feeling_overwhelmed': ["Đừng lo! Hãy để tôi giúp bạn sắp xếp lại ưu tiên nhé."],
    'feeling_productive': ["Tuyệt vời! Bạn đang làm rất tốt!"],
    'motivation': ["Mỗi bước nhỏ đều đưa bạn đến gần mục tiêu hơn! Cố lên!"],
    'out_of_scope': ["Xin lỗi, tôi chỉ hỗ trợ về quản lý công việc."],
}

def get_response(text):
    tag, confidence, source = predict_3tier(text)
    task_name = extract_task_name(text, tag)

    if confidence < 0.5:
        return f'[{source} {confidence:.1%}] Xin lỗi, tôi chưa hiểu ý bạn.'

    templates = RESPONSE_TEMPLATES.get(tag, ["Tôi đã hiểu yêu cầu của bạn."])
    if task_name:
        tw = [t for t in templates if '{task_name}' in t]
        response = random.choice(tw).format(task_name=task_name) if tw else random.choice(templates)
    else:
        tw = [t for t in templates if '{task_name}' not in t]
        response = random.choice(tw) if tw else random.choice(templates).replace("'{task_name}'", "công việc")

    return f'[{source} {confidence:.1%}] {response}'

# Test hệ thống 3 tầng
test_messages = [
    'xin chào',
    'thêm công việc họp với sếp',
    'giúp tôi thêm công việc: hoàn thành bài tập về nhà',
    'tạo task mới viết báo cáo tháng 4',
    'hoàn thành Viết báo cáo',
    'Sửa lỗi critical đã hoàn thành',
    'tôi đã hoàn thành công việc Tối ưu hóa ứng dụng',
    'hôm nay tôi có gì',
    'xóa việc đi chợ',
    'kiểm tra danh sách công việc',
    'nhắc tôi họp nhóm lúc 3 giờ',
    'cảm ơn bạn nhiều',
    'bye bye',
    'tôi có quá nhiều việc',
    'bạn là ai?',
    'thời tiết hôm nay thế nào?',
    'done task review code',
    'việc nào bị trễ hạn?',
    'ý tôi là công việc tên là hoàn thành bài tập',
]

print('=== TEST HỆ THỐNG 3 TẦNG ===\n')
for msg in test_messages:
    print(f'User: {msg}')
    print(f'Bot:  {get_response(msg)}\n')

## 9.1 Chat Interactive

Chạy cell bên dưới để chat trực tiếp với chatbot. Gõ `quit` hoặc `thoát` để dừng.

In [ ]:
# Bộ nhớ tạm lưu công việc
task_list = []
task_id_counter = 0
pending_action = None
waiting_for = None

def handle_action(tag, task_name, user_input):
    global task_list, task_id_counter, pending_action
    if tag == 'create_task' and task_name:
        task_id_counter += 1
        task_list.append({'id': task_id_counter, 'name': task_name, 'status': 'chưa bắt đầu'})
        return f"Đã tạo công việc '{task_name}' (ID: {task_id_counter})."
    elif tag == 'list_tasks':
        if not task_list: return "Bạn chưa có công việc nào. Hãy thêm công việc mới!"
        lines = ["Danh sách công việc:"]
        for t in task_list:
            icon = "✅" if t['status'] == 'hoàn thành' else "⬜"
            lines.append(f"  {icon} {t['id']}. {t['name']} [{t['status']}]")
        return "\n".join(lines)
    elif tag == 'status_update' and task_name:
        for t in task_list:
            if task_name.lower() in t['name'].lower():
                t['status'] = 'hoàn thành'
                return f"Đã đánh dấu '{t['name']}' hoàn thành! ✅"
        return f"Không tìm thấy công việc '{task_name}'."
    elif tag == 'delete_task' and task_name:
        for t in task_list:
            if task_name.lower() in t['name'].lower():
                pending_action = {'type': 'delete', 'task': t}
                return f"Bạn có chắc muốn xóa '{t['name']}'? (Có/Không)"
        return f"Không tìm thấy công việc '{task_name}'."
    elif tag == 'complete_all_tasks':
        if not task_list: return "Không có công việc nào."
        pending_action = {'type': 'complete_all'}
        count = sum(1 for t in task_list if t['status'] != 'hoàn thành')
        return f"Đánh dấu hoàn thành TẤT CẢ {count} công việc? (Có/Không)"
    elif tag == 'delete_all_tasks':
        if not task_list: return "Không có công việc nào."
        pending_action = {'type': 'delete_all'}
        return f"⚠️ XÓA TẤT CẢ {len(task_list)} công việc? (Có/Không)"
    elif tag == 'statistics':
        if not task_list: return "Chưa có công việc nào."
        total = len(task_list)
        done = sum(1 for t in task_list if t['status'] == 'hoàn thành')
        pct = done / total * 100 if total > 0 else 0
        return f"Thống kê: {total} tổng | {done} xong ({pct:.0f}%) | {total-done} chưa"
    elif tag == 'task_today':
        if not task_list: return "Hôm nay chưa có việc gì."
        pending = [t for t in task_list if t['status'] != 'hoàn thành']
        if not pending: return "Đã hoàn thành tất cả!"
        lines = ["Công việc hôm nay:"]
        for t in pending: lines.append(f"  ⬜ {t['id']}. {t['name']}")
        return "\n".join(lines)
    return None

print('=== CHATBOT 3 TẦNG (LSTM + PhoBERT) ===')
print("Gõ 'quit' hoặc 'thoát' để dừng.\n")

while True:
    user_input = input('Bạn: ')
    if user_input.lower().strip() in ['quit', 'thoát', 'exit', 'q']:
        print('Bot: Tạm biệt!')
        break
    if not user_input.strip(): continue
    lower = user_input.lower().strip()

    # 1. Chờ xác nhận
    if pending_action:
        if lower in ['có', 'co', 'yes', 'ừ', 'ok', 'đồng ý']:
            if pending_action['type'] == 'complete_all':
                c = sum(1 for t in task_list if t['status'] != 'hoàn thành')
                for t in task_list: t['status'] = 'hoàn thành'
                pending_action = None
                print(f'Bot: Đã hoàn thành {c} công việc! ✅\n')
            elif pending_action['type'] == 'delete':
                t = pending_action['task']; task_list.remove(t); pending_action = None
                print(f"Bot: Đã xóa '{t['name']}'.\n")
            elif pending_action['type'] == 'delete_all':
                c = len(task_list); task_list.clear(); pending_action = None
                print(f'Bot: Đã xóa {c} công việc.\n')
            continue
        elif lower in ['không', 'no', 'thôi', 'hủy']:
            pending_action = None
            print('Bot: Đã hủy.\n')
            continue

    # 2. Chờ nhập tên task
    if waiting_for:
        tag = waiting_for; waiting_for = None
        r = handle_action(tag, user_input.strip(), user_input)
        print(f"Bot: {r or 'Đã ghi nhận.'}\n")
        continue

    # 3. Hệ thống 3 tầng
    tag, confidence, source = predict_3tier(user_input)
    task_name = extract_task_name(user_input, tag)

    if confidence < 0.5:
        print(f'Bot: Xin lỗi, tôi chưa hiểu. Bạn có thể nói rõ hơn?')
        print(f'     [{source} {confidence:.1%}]\n')
        continue

    # 4. Cần tên task?
    if tag in ['create_task', 'delete_task', 'status_update', 'update_task'] and not task_name:
        waiting_for = tag
        prompts = {'create_task': "Tên công việc:", 'delete_task': "Xóa việc nào:",
                   'status_update': "Hoàn thành việc nào:", 'update_task': "Sửa việc nào:"}
        print(f'Bot: {prompts[tag]}')
        print(f'     [{source} {tag} {confidence:.1%}]\n')
        continue

    # 5. Action
    r = handle_action(tag, task_name, user_input)
    if r:
        response = r
    else:
        templates = RESPONSE_TEMPLATES.get(tag, ["Tôi hiểu rồi."])
        if task_name:
            tw = [t for t in templates if '{task_name}' in t]
            response = random.choice(tw).format(task_name=task_name) if tw else random.choice(templates)
        else:
            tw = [t for t in templates if '{task_name}' not in t]
            response = random.choice(tw) if tw else random.choice(templates).replace("'{task_name}'", "công việc")

    ent = f'task="{task_name}"' if task_name else ''
    print(f'Bot: {response}')
    print(f'     [{source} {tag} {confidence:.1%} {ent}]\n')

## 10. Lưu Model & Tải về

In [ ]:
# === Lưu LSTM model ===
torch.save(model.state_dict(), 'chatbot_model.pth')
print('1. LSTM model saved: chatbot_model.pth')

# === Lưu PhoBERT model ===
torch.save(phobert_model.state_dict(), 'phobert_model.pth')
phobert_tokenizer.save_pretrained('phobert_tokenizer')
print('2. PhoBERT model saved: phobert_model.pth')
print('3. PhoBERT tokenizer saved: phobert_tokenizer/')

# === Lưu metadata ===
metadata = {
    'word2idx': word2idx,
    'tags': tags,
    'vocab_size': VOCAB_SIZE,
    'embedding_dim': EMBEDDING_DIM,
    'hidden_dim': HIDDEN_DIM,
    'num_layers': NUM_LAYERS,
    'dropout': DROPOUT,
    'max_len': MAX_LEN,
    'phobert_name': phobert_name,
    'phobert_hidden_size': phobert_base.config.hidden_size,
    'lstm_threshold': 0.85,
    'phobert_threshold': 0.60,
}
with open('model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print('4. Metadata saved: model_metadata.json')

# === Lưu responses ===
with open('responses.json', 'w', encoding='utf-8') as f:
    json.dump(responses_dict, f, ensure_ascii=False, indent=2)
print('5. Responses saved: responses.json')

print('\n--- Tải các file sau về và đặt vào backend/data/ ---')
print('1. chatbot_model.pth       (LSTM)')
print('2. phobert_model.pth       (PhoBERT)')
print('3. phobert_tokenizer/      (PhoBERT tokenizer)')
print('4. model_metadata.json     (config)')
print('5. responses.json          (responses)')

In [ ]:
# Tải file về máy (Google Colab)
import shutil
from google.colab import files

# Nén tokenizer thành zip
shutil.make_archive('phobert_tokenizer', 'zip', '.', 'phobert_tokenizer')

files.download('chatbot_model.pth')
files.download('phobert_model.pth')
files.download('phobert_tokenizer.zip')
files.download('model_metadata.json')
files.download('responses.json')